# Assets

Create, inspect, upload, and download precomputed AlphaEdit assets.

Asset creation is intentionally guarded by `CREATE_*` flags because it is a heavy GPU setup job. Benchmark datasets are loaded directly from HuggingFace in `generation.ipynb`, so this notebook only manages method assets.

## Environment And Config

In [ ]:
from __future__ import annotations

import gc
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import torch
import yaml
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

load_dotenv()


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.editors import load_editor
from src.editors.alpha_edit.alpha_edit_main import get_project
from src.utils import nethook

EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

model_name = eval_config["model_name"]
REPO_ID = "Savassan/KnowledgeSurgeon"
REMOTE_BASE = f"artifacts/{model_name}"

ALPHA_EDIT_CONFIG = PROJECT_ROOT / "src" / "config" / "alpha_edit" / f"{model_name}.yaml"

PROJECTORS_DIR = PROJECT_ROOT / "data" / "alpha_edit_projectors"
LAYER_STATS_DIR = PROJECT_ROOT / "data" / "layer_stats"
ALPHA_EDIT_PROJECTOR = PROJECTORS_DIR / "null_space_project.pt"
TELEMETRY_DIR = PROJECT_ROOT / "data" / "evaluation" / model_name / "setup"

ASSET_ROOTS = (
    PROJECTORS_DIR,
    LAYER_STATS_DIR,
)

CREATE_PROJECTOR = False
FORCE_RECREATE_COVARIANCE = False
FORCE_RECREATE_PROJECTOR = True

## Local And Remote Inspection

In [ ]:
def iter_method_asset_files() -> list[Path]:
    files: list[Path] = []

    if ALPHA_EDIT_PROJECTOR.exists():
        files.append(ALPHA_EDIT_PROJECTOR)

    if LAYER_STATS_DIR.exists():
        files.extend(sorted(LAYER_STATS_DIR.rglob("*.npz")))

    return sorted(dict.fromkeys(files))


def remote_asset_path(local_file: Path) -> str:
    relative_path = local_file.relative_to(PROJECT_ROOT).as_posix()
    return f"{REMOTE_BASE}/{relative_path}"


def local_asset_path(remote_file: str) -> Path:
    relative_path = remote_file.removeprefix(f"{REMOTE_BASE}/")
    return PROJECT_ROOT / Path(relative_path)


def summarize_local_assets() -> dict:
    files = iter_method_asset_files()
    return {
        "count": len(files),
        "total_bytes": sum(path.stat().st_size for path in files),
        "projector_exists": ALPHA_EDIT_PROJECTOR.exists(),
        "layer_stats_count": sum(1 for path in files if path.suffix == ".npz"),
        "files": [path.relative_to(PROJECT_ROOT).as_posix() for path in files],
    }


def list_remote_method_assets() -> list[str]:
    api = HfApi()
    prefixes = tuple(
        f"{REMOTE_BASE}/{asset_root.relative_to(PROJECT_ROOT).as_posix()}/"
        for asset_root in ASSET_ROOTS
    )
    return sorted(
        path
        for path in api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
        if path.startswith(prefixes)
    )

## Create Assets

This section is safe on `Run All` while `CREATE_PROJECTOR` is `False`.

Set `CREATE_PROJECTOR = True` to explicitly build the AlphaEdit null-space projector. The projector is forced by default because its filename does not encode the `mom2_n_samples` setting. `FORCE_RECREATE_COVARIANCE` controls whether projector creation also rebuilds the underlying layer statistics.

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def clear_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def file_size(path: Path) -> int | None:
    return path.stat().st_size if path.exists() else None


def layer_progress(desc: str):
    return lambda iterator, total, leave=False: tqdm(iterator, total=total, leave=leave, desc=desc)


def projector_shape(model, hparams) -> tuple[int, int, int]:
    weight = nethook.get_parameter(
        model,
        f"{hparams.rewrite_module_tmp.format(hparams.layers[-1])}.weight",
    )
    lower_model_name = hparams.model_name.lower()
    if "llama" in lower_model_name or "qwen" in lower_model_name or "gpt-j-6b" in lower_model_name:
        dim = weight.shape[1]
    elif "gpt2-xl" in lower_model_name:
        dim = weight.shape[0]
    else:
        raise ValueError(f"Unsupported model for AlphaEdit projector: {hparams.model_name}")
    return (len(hparams.layers), dim, dim)

In [ ]:
def build_alpha_projector(
    model,
    tok,
    hparams,
    force_recompute: bool,
    covariance_force_recompute: bool = False,
) -> dict:
    p_loc = Path(hparams.P_loc)
    report = {
        "config_path": str(ALPHA_EDIT_CONFIG),
        "layers": [],
        "P_loc": str(p_loc),
        "mom2_dataset": hparams.mom2_dataset,
        "mom2_n_samples": hparams.mom2_n_samples,
        "mom2_dtype": hparams.mom2_dtype,
        "nullspace_threshold": hparams.nullspace_threshold,
        "covariance_force_recompute": covariance_force_recompute,
        "existed_before": p_loc.exists(),
    }

    if p_loc.exists() and not force_recompute:
        report.update({
            "status": "skipped_existing",
            "exists_after": True,
            "file_size_bytes": file_size(p_loc),
            "elapsed_seconds": 0.0,
        })
        return report

    start = perf_counter()
    P = torch.zeros(projector_shape(model, hparams), device="cpu")
    with tqdm(total=len(hparams.layers), desc="AlphaEdit projector") as projector_pbar:
        for i, layer in enumerate(hparams.layers):
            layer_start = perf_counter()
            layer_name = hparams.rewrite_module_tmp.format(layer)

            def set_project_status(status: str, layer_name: str = layer_name):
                projector_pbar.set_postfix_str(f"{layer_name}: {status}", refresh=True)

            P[i, :, :] = get_project(
                model,
                tok,
                layer,
                hparams,
                progress=layer_progress(f"{layer_name} batches"),
                force_recompute=covariance_force_recompute,
                status_callback=set_project_status,
            )
            sync_cuda()
            clear_cuda_cache()
            projector_pbar.update(1)
            report["layers"].append({
                "layer": layer,
                "elapsed_seconds": perf_counter() - layer_start,
            })

    p_loc.parent.mkdir(parents=True, exist_ok=True)
    torch.save(P, p_loc)
    del P
    clear_cuda_cache()
    report.update({
        "status": "created" if not report["existed_before"] else "recreated",
        "exists_after": p_loc.exists(),
        "file_size_bytes": file_size(p_loc),
        "elapsed_seconds": perf_counter() - start,
    })
    return report

In [ ]:
def create_method_assets(
    create_projector: bool = CREATE_PROJECTOR,
    force_recreate_covariance: bool = FORCE_RECREATE_COVARIANCE,
    force_recreate_projector: bool = FORCE_RECREATE_PROJECTOR,
) -> dict | None:
    if not create_projector:
        print("Asset creation is disabled. Set CREATE_PROJECTOR to True to run it.")
        return None

    alpha_hparams = load_config(ALPHA_EDIT_CONFIG)
    TELEMETRY_DIR.mkdir(parents=True, exist_ok=True)
    load_start = perf_counter()
    editor = load_editor(str(ALPHA_EDIT_CONFIG))
    model = editor.model
    tok = editor.tokenizer
    model.eval()
    sync_cuda()
    model_load_elapsed = perf_counter() - load_start

    run_start = perf_counter()
    telemetry = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model_key": model_name,
        "model_name": alpha_hparams.model_name,
        "force_recreate": {
            "covariance": force_recreate_covariance,
            "projector": force_recreate_projector,
        },
        "requested_assets": {
            "projector": create_projector,
        },
        "model_load_elapsed_seconds": model_load_elapsed,
        "assets": {},
    }

    try:
        if create_projector:
            telemetry["assets"]["alpha_edit_projector"] = build_alpha_projector(
                model,
                tok,
                alpha_hparams,
                force_recompute=force_recreate_projector,
                covariance_force_recompute=force_recreate_covariance,
            )
    finally:
        del editor, model, tok
        clear_cuda_cache()

    telemetry["total_elapsed_seconds"] = perf_counter() - run_start
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    telemetry_path = TELEMETRY_DIR / f"method_assets.{timestamp}.json"
    latest_path = TELEMETRY_DIR / "method_assets.latest.json"
    for path in [telemetry_path, latest_path]:
        with open(path, "w", encoding="utf-8") as file:
            json.dump(telemetry, file, indent=2)
    print(f"Saved telemetry: {telemetry_path}")
    print(f"Saved latest telemetry: {latest_path}")
    return telemetry

## Upload And Download Assets

In [ ]:
def upload_method_assets() -> None:
    files_to_upload = iter_method_asset_files()
    if not files_to_upload:
        raise FileNotFoundError(
            "No method assets found. Expected AlphaEdit projector and/or layer_stats .npz files."
        )

    api = HfApi()
    for local_file in files_to_upload:
        relative_path = local_file.relative_to(PROJECT_ROOT).as_posix()
        api.upload_file(
            path_or_fileobj=str(local_file),
            path_in_repo=remote_asset_path(local_file),
            repo_id=REPO_ID,
            repo_type="dataset",
        )
        print(f"Uploaded {relative_path}")


def download_method_assets(overwrite: bool = False) -> None:
    remote_files = list_remote_method_assets()
    if not remote_files:
        raise FileNotFoundError(f"No method assets found under {REPO_ID}/{REMOTE_BASE}")

    for remote_file in remote_files:
        destination = local_asset_path(remote_file)
        relative_path = destination.relative_to(PROJECT_ROOT).as_posix()
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists() and not overwrite:
            print(f"Skipped {relative_path}")
            continue

        cached_file = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=remote_file,
            force_download=overwrite,
        )
        shutil.copy2(cached_file, destination)
        print(f"Downloaded {relative_path}")

## Manual Actions

In [ ]:
# Inspect local files that would be uploaded:
# summarize_local_assets()

# Inspect remote files available for download:
# list_remote_method_assets()

# Create method assets after flipping CREATE_* flags above:
# create_method_assets()

# Upload method assets from the local machine:
# upload_method_assets()

# Download method assets on a rented machine:
# download_method_assets()

# Re-download and replace local method assets if needed:
# download_method_assets(overwrite=True)